In [0]:
# Environment configuration - reads from job parameter or defaults to dev
import os

# Get environment from widget (job parameter) or default to 'dev'
dbutils.widgets.text("environment", "dev", "Environment (dev/prod)")
environment = dbutils.widgets.get("environment")

print(f"Running in {environment.upper()} environment")
print(f"Table: {environment}.stock_daily_data")

Running in DEV environment
Table: dev.stock_daily_data


In [0]:
%pip install yfinance

import yfinance as yf
import pandas as pd

# Get tickers from widget parameter
dbutils.widgets.text("tickers", "RELIANCE.NS,TCS.NS,INFY.NS", "Stock Tickers (comma-separated)")
tickers_param = dbutils.widgets.get("tickers")
tickers = [ticker.strip() for ticker in tickers_param.split(",")]
data = yf.download(tickers, period="1d", interval="1d")

# Restructure: transpose to have stocks as rows and price metrics as columns
results = []
if len(tickers) == 1:
    # Single ticker: data has single-level columns
    ticker = tickers[0]
    row = {
        'Stock': ticker,
        'Open': data['Open'].values[0],
        'High': data['High'].values[0],
        'Low': data['Low'].values[0],
        'Close': data['Close'].values[0],
        'Volume': data['Volume'].values[0]
    }
    results.append(row)
else:
    # Multiple tickers: data has multi-level columns (metric, ticker)
    for ticker in tickers:
        row = {
            'Stock': ticker,
            'Open': data[('Open', ticker)].values[0],
            'High': data[('High', ticker)].values[0],
            'Low': data[('Low', ticker)].values[0],
            'Close': data[('Close', ticker)].values[0],
            'Volume': data[('Volume', ticker)].values[0]
        }
        results.append(row)

df_final = pd.DataFrame(results)

# Display
display(df_final)

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


[*********************100%***********************]  2 of 3 completed


Stock,Open,High,Low,Close,Volume
RELIANCE.NS,1304.0999755859375,1306.800048828125,1288.0,1294.9000244140625,9797263
TCS.NS,2270.0,2274.5,2244.0,2255.5,2142654
INFY.NS,1095.0,1095.0,1078.9000244140625,1082.0,6262990


In [0]:
# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df_final)

# Add date column for tracking
from pyspark.sql.functions import current_date
spark_df = spark_df.withColumn('fetch_date', current_date())

# Define table name
table_name = f'{environment}.stock_daily_data'

# Write to Delta table (will create if doesn't exist, append if exists)
spark_df.write.format('delta') \
    .mode('append') \
    .saveAsTable(table_name)

print(f'Successfully saved {spark_df.count()} records to table: {table_name}')

Successfully saved 3 records to table: stock_daily_data


In [0]:
%sql
select * from ${environment}.stock_daily_data order by fetch_date desc;

Stock,Open,High,Low,Close,Volume,fetch_date
RELIANCE.NS,1334.0,1342.0,1318.699951171875,1335.9000244140625,13021537,2026-05-18
TCS.NS,2264.0,2292.89990234375,2252.199951171875,2283.199951171875,3772966,2026-05-18
INFY.NS,1119.0,1146.0999755859375,1118.0,1142.5,12095591,2026-05-18
